#1.ARQUIVOS API

## Passo 1 · Extrair: pedir os feriados à API

In [0]:
import requests                # conversa com a API (HTTP)

url = "https://brasilapi.com.br/api/feriados/v1/2026"
resposta = requests.get(url, timeout=30)
resposta.raise_for_status()    # se a API cair, para aqui com o erro, em vez de seguir com nada

feriados = resposta.json()     # o JSON vira uma lista do Python: um dicionário por feriado
print(len(feriados), "feriados")
feriados[:3]

## Passo 2 · Pousar: o JSON numa pasta nova do raw

In [0]:
import json, os
from datetime import datetime

pasta = "/Volumes/lakehouse_vertice/bronze/raw/feriados"
os.makedirs(pasta, exist_ok=True)                        # a pasta nova precisa existir

agora = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")     # ex.: 2026-10-20_19-40-00 (horário UTC)
with open(f"{pasta}/feriados_{agora}.json", "w") as arquivo:
    json.dump(feriados, arquivo, ensure_ascii=False)

sorted(os.listdir(pasta))                                # um arquivo por execução

## Passo 3 · Carregar: o raw vira a **bronze**

In [0]:
 %sql
 CREATE OR REPLACE TABLE lakehouse_vertice.bronze.feriados
 COMMENT 'Feriados nacionais de 2026, da BrasilAPI. Uma captura por execução do job.'
 AS SELECT date AS data, name AS feriado, type AS tipo, weekday AS dia_da_semana,
           _metadata.file_name              AS arquivo,
           _metadata.file_modification_time AS coletado_em
 FROM read_files('/Volumes/lakehouse_vertice/bronze/raw/feriados/', format => 'json', multiLine => true)

## Passo 4 · Conferir

In [0]:
 %sql
 -- uma linha por captura: 14 feriados em cada uma
SELECT arquivo, coletado_em, COUNT(*) AS feriados
FROM lakehouse_vertice.bronze.feriados
GROUP BY arquivo, coletado_em
ORDER BY coletado_em DESC

Final

In [0]:
 %sql
-- a resposta para o negócio, na captura mais recente:
 -- 02/11 (segunda), 15/11 (domingo) e 20/11 (sexta): 2 dias úteis a menos em novembro
SELECT data, feriado, dia_da_semana
FROM lakehouse_vertice.bronze.feriados
WHERE arquivo = (SELECT MAX(arquivo) FROM lakehouse_vertice.bronze.feriados)
AND data BETWEEN '2026-11-01' AND '2026-11-30'
ORDER BY data